# 02 v2 — Fine-Tune NaijaReviewer-8B + GGUF + Push (all-in-one)

End-to-end: QLoRA fine-tune → merge → smoke-test → push HF model → GGUF convert → quantize → push GGUF + Modelfile + README.

## What changed vs v1

Two bugs that mattered:

1. **EOS token now appended to training text.** Without it the model never learns to stop generating — it rambles past the answer in production.
2. **Response-only loss via `train_on_responses_only`.** Loss is now computed only on `### Response` tokens instead of the whole sequence. Typically gives a noticeable quality bump because model capacity isn't wasted on the instruction scaffolding.

Plus quality-of-life:
- Truncation diagnostic before training (catches silent length issues)
- Verification step that masking is actually working
- Separate `naija-reviewer-v2` checkpoint dir — **do not** resume from v1 checkpoints, the data format changed
- GGUF conversion + quantization (Q4_K_M / Q5_K_M / Q8_0) right in this notebook
- Auto-generated Modelfile (with the correct Alpaca template) + model card README
- All artifacts pushed to one HF repo

## Runtime
- GPU runtime required for fine-tuning (A100 / L4 / T4)
- GGUF section is CPU-bound — runs fine on the same GPU runtime
- **End-to-end**: ~2.5h on A100, ~4h on L4, ~7h on T4
- **Disk**: needs ~50 GB free locally (merged HF + f16 GGUF + quants)

## Inputs
- `HF_TOKEN` — needs **write** access (download base model + push everything)
- `ANTHROPIC_API_KEY` — optional, eval baseline only

Accept the Llama 3.1 license: [meta-llama/Meta-Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct)


## 0. Setup


In [ ]:
# Unsloth-only install — clean stack
!pip install -q --upgrade pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.13" peft accelerate bitsandbytes wandb sentencepiece
!pip install -q pandas jsonlines pyarrow datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 46.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch
from unsloth import FastLanguageModel
print(f"✅ torch {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print("✅ unsloth ready")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ torch 2.10.0+cu128 (CUDA: True)
✅ unsloth ready


In [ ]:
# Drive mount + paths — NOTE: v2 uses its own checkpoint dir
import os, sys, json, gc, time, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_DIR_OPENAI = Path("/content/drive/MyDrive/naija_persona_corpus_openai")
    DRIVE_DIR_DD     = Path("/content/drive/MyDrive/naija_persona_corpus")
    DRIVE_DIR        = Path("/content/drive/MyDrive/naija_persona_corpus_merged_runs")
else:
    DRIVE_DIR_OPENAI = Path.home() / "naija_persona_corpus_openai"
    DRIVE_DIR_DD     = Path.home() / "naija_persona_corpus"
    DRIVE_DIR        = Path.home() / "naija_persona_corpus-merged-runs"

CORPUS_DIR_OPENAI = DRIVE_DIR_OPENAI / "processed"
CORPUS_DIR_DD     = DRIVE_DIR_DD / "processed"

# ⚠️ v2 — separate checkpoint dir so we don't accidentally resume from v1 (different data format)
TRAINING_CKPT = DRIVE_DIR / "checkpoints" / "naija-reviewer-v2"
MERGED_DIR    = DRIVE_DIR / "merged" / "naija-reviewer-8b-v2"
RESULTS_DIR   = DRIVE_DIR / "results"

# Local workspace for fast I/O (GGUF conversion, llama.cpp build)
WORK_DIR      = Path("/content/gguf_work" if IN_COLAB else Path.home() / "gguf_work")
MERGED_LOCAL  = WORK_DIR / "merged_hf"
GGUF_OUT_DIR  = WORK_DIR / "gguf_out"

for d in (TRAINING_CKPT, MERGED_DIR, RESULTS_DIR, WORK_DIR, GGUF_OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# HF repo config — bump the suffix or remove it to overwrite the v1 repo
HF_ORG     = "Shinzmann"
MODEL_REPO = f"{HF_ORG}/naija-reviewer-8b-v2"      # main repo (HF + GGUF)
ADAPTER_REPO = f"{HF_ORG}/naija-reviewer-8b-v2-lora"
DATASET_REPO = f"{HF_ORG}/npa-corpus-v1"
MODEL_NAME = "naija-reviewer-8b-v2"

print(f"✅ Drive:      {DRIVE_DIR}")
print(f"✅ Checkpoint: {TRAINING_CKPT}")
print(f"✅ Merged:     {MERGED_DIR}")
print(f"✅ Work dir:   {WORK_DIR}")
print(f"✅ HF repo:    {MODEL_REPO}")

Mounted at /content/drive
✅ Drive:      /content/drive/MyDrive/naija_persona_corpus_merged_runs
✅ Checkpoint: /content/drive/MyDrive/naija_persona_corpus_merged_runs/checkpoints/naija-reviewer-v2
✅ Merged:     /content/drive/MyDrive/naija_persona_corpus_merged_runs/merged/naija-reviewer-8b-v2
✅ Work dir:   /content/gguf_work
✅ HF repo:    Shinzmann/naija-reviewer-8b-v2


In [ ]:
# Secrets — HF (required, needs write) + Anthropic (optional)
def _set_secret(name, prompt, required=True):
    if IN_COLAB:
        from google.colab import userdata
        try:
            v = userdata.get(name)
            if v:
                os.environ[name] = v
                print(f"  ✅ {name} from Colab Secrets")
                return v
        except Exception:
            pass
    v = os.environ.get(name)
    if not v and required:
        from getpass import getpass
        v = getpass(prompt); os.environ[name] = v
    if v: print(f"  ✅ {name} configured")
    elif required: raise RuntimeError(f"{name} required")
    return v

_set_secret("HF_TOKEN", "HF_TOKEN (hf_…, needs write): ", required=True)
_set_secret("ANTHROPIC_API_KEY", "ANTHROPIC_API_KEY (optional): ", required=False)

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

  ✅ HF_TOKEN from Colab Secrets
  ✅ ANTHROPIC_API_KEY from Colab Secrets


## 1. GPU Autotune


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("No GPU — Runtime → Change runtime type → A100/L4/H100/T4")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
BF16_OK  = torch.cuda.is_bf16_supported()

name_l = GPU_NAME.lower()
if   "h100" in name_l:                  CFG = {"batch":16,"grad_accum":2, "workers":4,"seq_len":4096}
elif "a100" in name_l and GPU_VRAM>60:  CFG = {"batch":8, "grad_accum":4, "workers":4,"seq_len":4096}
elif "a100" in name_l:                  CFG = {"batch":4, "grad_accum":4, "workers":4,"seq_len":4096}
elif "l4"   in name_l:                  CFG = {"batch":2, "grad_accum":16,"workers":2,"seq_len":3072}
elif "t4"   in name_l:                  CFG = {"batch":1, "grad_accum":32,"workers":2,"seq_len":2048}
else:                                   CFG = {"batch":2, "grad_accum":16,"workers":2,"seq_len":3072}

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

print(f"GPU: {GPU_NAME} ({GPU_VRAM:.1f}GB)  bf16={BF16_OK}")
print(f"Autotuned: batch={CFG['batch']} × grad_accum={CFG['grad_accum']} = {CFG['batch']*CFG['grad_accum']} effective")
print(f"           seq_len={CFG['seq_len']}, workers={CFG['workers']}")

GPU: NVIDIA A100-SXM4-40GB (42.4GB)  bf16=True
Autotuned: batch=4 × grad_accum=4 = 16 effective
           seq_len=4096, workers=4


## 2. Load Model + Dataset

**Note**: v2 changes the training-data format (appends EOS). Resuming from a v1 checkpoint will pollute the run — keep `TRAINING_CKPT` pointed at the v2 dir.


In [ ]:
# Resume detection (v2 dir only — see note above)
checkpoints = sorted(TRAINING_CKPT.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
RESUME_FROM = checkpoints[-1] if checkpoints else None
print(f"resume: {RESUME_FROM.name if RESUME_FROM else 'fresh start'}")

resume: checkpoint-1066


In [ ]:
# Load Llama 3.1 8B via Unsloth
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

if RESUME_FROM:
    print(f"Loading from checkpoint {RESUME_FROM.name}...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(RESUME_FROM),
        max_seq_length=CFG["seq_len"], dtype=None, load_in_4bit=True,
    )
    FastLanguageModel.for_training(model)
else:
    print("Loading Llama 3.1 8B (Unsloth pre-quantized 4-bit, ~5GB)...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
        max_seq_length=CFG["seq_len"], dtype=None, load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=LORA_R, target_modules=LORA_TARGETS,
        lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias="none", use_gradient_checkpointing="unsloth", random_state=42,
    )

# Padding side: "right" for training. If you batch-generate later, switch to "left".
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n✅ trainable: {trainable/1e6:.1f}M params")
print(f"✅ EOS token: {tokenizer.eos_token!r}  (id={tokenizer.eos_token_id})")

Loading from checkpoint checkpoint-1066...
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.5.2 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.



✅ trainable: 41.9M params
✅ EOS token: '<|eot_id|>'  (id=128009)


In [ ]:
# Load + format both corpora — KEY FIX: append EOS to every example
from datasets import Dataset, concatenate_datasets

EOS = tokenizer.eos_token   # captured once; consistent across the dataset

def alpaca_to_text(r):
    # 🔑 v2 FIX: append EOS so the model learns to stop.
    return {"text":
        f"### Instruction\n{r['instruction']}\n\n"
        f"### Input\n{r['input']}\n\n"
        f"### Response\n{r['output']}{EOS}"
    }

def load_and_format_corpus(corpus_dir):
    train_path = corpus_dir / "v1_train_alpaca.jsonl"
    val_path = corpus_dir / "v1_val_alpaca.jsonl"
    if not train_path.exists() or not val_path.exists():
        print(f"⚠️ Missing files in {corpus_dir}, skipping...")
        return None, None
    train_raw = [json.loads(l) for l in train_path.open()]
    val_raw   = [json.loads(l) for l in val_path.open()]
    ds_train = Dataset.from_list(train_raw).map(alpaca_to_text)
    ds_val   = Dataset.from_list(val_raw).map(alpaca_to_text)
    return ds_train, ds_val

ds_train_openai, ds_val_openai = load_and_format_corpus(CORPUS_DIR_OPENAI)
ds_train_dd, ds_val_dd         = load_and_format_corpus(CORPUS_DIR_DD)

train_datasets = [ds for ds in [ds_train_openai, ds_train_dd] if ds is not None]
val_datasets   = [ds for ds in [ds_val_openai, ds_val_dd]     if ds is not None]
if not train_datasets:
    raise RuntimeError("❌ No corpora found — make sure 01_build_corpus.ipynb has been run.")

ds_train = concatenate_datasets(train_datasets).shuffle(seed=42)
ds_val   = concatenate_datasets(val_datasets)

print(f"\nMerged train={len(ds_train):,}  val={len(ds_val):,}")
print(f"\n─── sample (note EOS at end) ───")
print(repr(ds_train[0]["text"][-200:]))

Map:   0%|          | 0/5852 [00:00<?, ? examples/s]

Map:   0%|          | 0/322 [00:00<?, ? examples/s]

Map:   0%|          | 0/2676 [00:00<?, ? examples/s]

Map:   0%|          | 0/147 [00:00<?, ? examples/s]


Merged train=8,528  val=469

─── sample (note EOS at end) ───
'e 6s Plus 128GB and it arrived safely, no wahala at all. It’s still working fine till now, nna, and for the price I feel it was a good choice. Seller looks trusted biko, I’m happy with it."}<|eot_id|>'


In [ ]:
# 🔑 v2 NEW: truncation diagnostic — catches silent length issues before training
import numpy as np

print("📏 checking sequence lengths…")
N = min(2000, len(ds_train))   # sample to avoid slow tokenize on huge sets
sample_idx = np.random.RandomState(42).choice(len(ds_train), N, replace=False)
lens = [len(tokenizer(ds_train[int(i)]["text"], add_special_tokens=True)["input_ids"])
        for i in sample_idx]

p50, p95, p99 = np.percentile(lens, [50, 95, 99])
over = sum(l > CFG["seq_len"] for l in lens)
pct  = 100 * over / len(lens)

print(f"   n={N}  mean={np.mean(lens):.0f}  p50={p50:.0f}  p95={p95:.0f}  p99={p99:.0f}  max={max(lens)}")
print(f"   seq_len={CFG['seq_len']}")
if pct > 5:
    print(f"   ⚠️  {over}/{N} ({pct:.1f}%) would be truncated — consider bumping seq_len")
    print(f"       or filter long examples before training")
else:
    print(f"   ✅ {over}/{N} ({pct:.1f}%) truncated — fine")

📏 checking sequence lengths…
   n=2000  mean=188  p50=180  p95=244  p99=277  max=452
   seq_len=4096
   ✅ 0/2000 (0.0%) truncated — fine


In [ ]:
# W&B (optional)
import wandb
try:
    wandb.init(project="naija_persona_corpus", name="naija-reviewer-8b-v2",
               config={"gpu": GPU_NAME, "vram": GPU_VRAM, **CFG, "version": "v2"},
               mode="online", resume="allow")
    print("✅ W&B online")
except Exception as e:
    print(f"⚠️ W&B disabled: {e}")
    os.environ["WANDB_DISABLED"] = "true"

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: chidi-ashinze (naija-petro) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


✅ W&B online


## 3. Train (response-only loss)

🔑 **v2 FIX**: wrap the trainer with `train_on_responses_only` so loss is only computed on `### Response\n…` tokens. Everything before that gets masked to `-100`. Verification step right after confirms the masking actually engaged.


In [ ]:
from transformers import DataCollatorForLanguageModeling
from trl import SFTTrainer, SFTConfig
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Pre-tokenize so we can drop the string columns (avoids the Unsloth+TRL prep collision)
def _tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=CFG["seq_len"],
                     padding=False, return_tensors=None)

print("🔤 pre-tokenizing…")
ds_train_tok = ds_train.map(_tokenize, batched=True, num_proc=4, desc="tokenizing train")
ds_val_tok   = ds_val.map(_tokenize,   batched=True, num_proc=4, desc="tokenizing val")

COLS_TO_DROP = [c for c in ["instruction", "input", "output", "text"] if c in ds_train_tok.column_names]
if COLS_TO_DROP:
    ds_train_tok = ds_train_tok.remove_columns(COLS_TO_DROP)
    ds_val_tok   = ds_val_tok.remove_columns(COLS_TO_DROP)

print(f"✅ columns: {ds_train_tok.column_names}")
print(f"   train: {len(ds_train_tok):,}  val: {len(ds_val_tok):,}")

🔤 pre-tokenizing…


tokenizing train (num_proc=4):   0%|          | 0/8528 [00:00<?, ? examples/s]

tokenizing val (num_proc=4):   0%|          | 0/469 [00:00<?, ? examples/s]

✅ columns: ['input_ids', 'attention_mask']
   train: 8,528  val: 469


In [ ]:
# Plain collator — train_on_responses_only will swap it for a masking version below
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8,
)

training_args = SFTConfig(
    output_dir=str(TRAINING_CKPT),
    num_train_epochs=2,
    per_device_train_batch_size=CFG["batch"],
    per_device_eval_batch_size=CFG["batch"],
    gradient_accumulation_steps=CFG["grad_accum"],
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    weight_decay=0.01,
    bf16=BF16_OK,
    fp16=not BF16_OK,
    optim="paged_adamw_8bit",
    logging_steps=25,
    save_strategy="steps",
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    save_total_limit=3,
    seed=42,
    report_to=["wandb"] if not os.environ.get("WANDB_DISABLED") else ["none"],
    run_name="naija-reviewer-8b-v2",
    max_seq_length=CFG["seq_len"],
    packing=False,
    dataset_text_field="text",
    dataset_kwargs={"skip_prepare_dataset": True},
    tf32=True,
    remove_unused_columns=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds_train_tok,
    eval_dataset=ds_val_tok,
    tokenizer=tokenizer,
    args=training_args,
    data_collator=data_collator,
)
print("✅ trainer built")

✅ trainer built


In [ ]:
# 🔑 v2 KEY FIX: response-only loss masking.
# We use TRL's DataCollatorForCompletionOnlyLM directly to avoid Unsloth list-tensor conflicts.
from trl import DataCollatorForCompletionOnlyLM
import torch

# Clean up any leftover 'labels' column from previous Unsloth runs
if "labels" in trainer.train_dataset.column_names:
    trainer.train_dataset = trainer.train_dataset.remove_columns("labels")
    trainer.eval_dataset = trainer.eval_dataset.remove_columns("labels")

trainer.data_collator = DataCollatorForCompletionOnlyLM(
    response_template="### Response\n",
    tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8,
)
print("✅ swapped collator → TRL's DataCollatorForCompletionOnlyLM")

# Verify the masking actually works — catch any tokenization mismatch BEFORE we burn GPU hours
print("\n🔍 verifying response-only masking…")

# Get the first item and pass it through the collator to generate the padded labels
item = trainer.train_dataset[0]
batch = trainer.data_collator([item])
labels = torch.as_tensor(batch["labels"][0])
input_ids = torch.as_tensor(batch["input_ids"][0])

n_unmasked = (labels != -100).sum().item()
n_total    = labels.numel()
pct = 100 * n_unmasked / n_total
print(f"   {n_unmasked}/{n_total} tokens have gradient ({pct:.1f}%)")

if n_unmasked == 0:
    raise RuntimeError(
        "❌ masking broken — response_template not found in tokenized data. "
        "Check that '### Response\\n' appears verbatim in your formatted text."
    )
if pct > 90:
    print("   ⚠️  >90% unmasked — masking may not be active. Inspect sample below.")

# Show what tokens are actually being trained on (should be the response text)
unmasked_text = tokenizer.decode(input_ids[labels != -100][:60])
print(f"\n   first unmasked tokens (model is trained on these):")
print(f"   {unmasked_text!r}")

✅ swapped collator → TRL's DataCollatorForCompletionOnlyLM

🔍 verifying response-only masking…
   68/168 tokens have gradient (40.5%)

   first unmasked tokens (model is trained on these):
   '{"rating": 4, "review": "I bought iPhone 6s Plus 128GB and it arrived safely, no wahala at all. It’s still working fine till now, nna, and for the price I feel it was a good choice. Seller looks trusted biko,'


In [ ]:
# Train
print("🚀 training…")
t0 = time.time()
if RESUME_FROM:
    trainer.train(resume_from_checkpoint=str(RESUME_FROM))
else:
    trainer.train()
print(f"\n✅ done in {(time.time()-t0)/60:.1f} min")

# Save adapter
FINAL_ADAPTER = TRAINING_CKPT / "final-adapter"
model.save_pretrained(str(FINAL_ADAPTER))
tokenizer.save_pretrained(str(FINAL_ADAPTER))
print(f"✅ adapter → {FINAL_ADAPTER}")

🚀 training…


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,528 | Num Epochs = 2 | Total steps = 1,066
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss,Validation Loss



✅ done in 0.2 min


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/naija_persona_corpus_merged_runs/checkpoints/naija-reviewer-v2/final-adapter/tokenizer_config.json.


✅ adapter → /content/drive/MyDrive/naija_persona_corpus_merged_runs/checkpoints/naija-reviewer-v2/final-adapter


In [ ]:
# Merge LoRA into base — save locally first (fast), then mirror to Drive
print("🔀 merging LoRA → 16-bit base…")
if MERGED_LOCAL.exists():
    shutil.rmtree(MERGED_LOCAL)
model.save_pretrained_merged(str(MERGED_LOCAL), tokenizer, save_method="merged_16bit")
print(f"✅ merged (local) → {MERGED_LOCAL}")

# Mirror to Drive for durability (GGUF conversion will use the local copy)
print(f"📤 mirroring to Drive…  (slow but durable)")
if MERGED_DIR.exists():
    shutil.rmtree(MERGED_DIR)
shutil.copytree(str(MERGED_LOCAL), str(MERGED_DIR))
print(f"✅ merged (drive) → {MERGED_DIR}")

🔀 merging LoRA → 16-bit base…


Unsloth: Restored added_tokens_decoder metadata in /content/gguf_work/merged_hf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:43<00:00, 10.94s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [00:55<00:00, 13.81s/it]


Unsloth: Merge process complete. Saved to `/content/gguf_work/merged_hf`
✅ merged (local) → /content/gguf_work/merged_hf
📤 mirroring to Drive…  (slow but durable)
✅ merged (drive) → /content/drive/MyDrive/naija_persona_corpus_merged_runs/merged/naija-reviewer-8b-v2


## 4. Smoke test (HF format)

Quick sanity check on a few val examples before we GGUF anything. Reloads the merged model in 16-bit on GPU.


In [ ]:
# Free training memory before reloading the merged model
try:    del trainer, model
except: pass
gc.collect(); torch.cuda.empty_cache()

naija_model, naija_tok = FastLanguageModel.from_pretrained(
    model_name=str(MERGED_LOCAL),
    max_seq_length=CFG["seq_len"], dtype=None, load_in_4bit=False,
)
FastLanguageModel.for_inference(naija_model)

CORPUS_DIR = CORPUS_DIR_OPENAI if (CORPUS_DIR_OPENAI / "v1_val_alpaca.jsonl").exists() else CORPUS_DIR_DD
ds_test_raw = [json.loads(l) for l in (CORPUS_DIR / "v1_val_alpaca.jsonl").open()]

print(f"🧪 smoke test — {len(ds_test_raw)} val samples available, showing 3\n")
for i in range(3):
    s = ds_test_raw[i]
    prompt = (f"### Instruction\n{s['instruction']}\n\n"
              f"### Input\n{s['input']}\n\n"
              f"### Response\n")
    inputs = naija_tok(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = naija_model.generate(
            **inputs, max_new_tokens=200, temperature=0.7,
            do_sample=True, pad_token_id=naija_tok.eos_token_id,
            eos_token_id=naija_tok.eos_token_id,   # explicit — uses our trained EOS
        )
    result = naija_tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    print(f"=== sample {i+1} ===")
    print(f"GT:    {s['output'][:200]}")
    print(f"MODEL: {result[:400]}\n")

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/gguf_work/merged_hf' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/gguf_work/merged_hf' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Will load /content/gguf_work/merged_hf as a legacy tokenizer.
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes

🧪 smoke test — 322 val samples available, showing 3



Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== sample 1 ===
GT:    {"rating": 4, "review": "I got it for my friend, and they are very happy with it. Honestly, na good one, because it has been working well for them, biko."}
MODEL: {"rating": 4, "review": "The phone is good and works well, no shaking. I’m quite happy with it, biko. It even came with a screen protector sef, so I can enjoy it without stress."}



Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== sample 2 ===
GT:    {"rating": 4, "review": "This baby product was exactly what I was expecting and it came right on time, nna. I’m really happy with it, biko, no disappointment at all. Ahn ahn, I’m already looking forwa
MODEL: {"rating": 4, "review": "Nna, we even use am for business calls too, biko. The battery dey last well well and it’s easy to carry around, no wahala. For the price, e make sense die, not that expensive at all."}

=== sample 3 ===
GT:    {"rating": 4, "review": "So far this phone dey perform well well, no wahala at all. Everything just dey work smooth like better Nollywood scene, abeg. I never see any serious issue, so na 4 star for n
MODEL: {"rating": 4, "review": "This phone na correct blockbuster, no cap. E come exactly as dem describe am, and I like say e no dey tied to any carrier, so I fit use am anywhere wey I need, no wahala. E dey work well and e no give me any stress at all, well done."}



In [ ]:
# Free GPU memory before GGUF (GGUF conversion is CPU-only)
try:    del naija_model, naija_tok
except: pass
gc.collect(); torch.cuda.empty_cache()
print(f"✅ GPU freed  (allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB)")

✅ GPU freed  (allocated: 1.06 GB)


## 5. Push HF-format model + adapter + dataset

GGUFs will go to the same `MODEL_REPO` later, so users get everything in one place.


In [ ]:
from huggingface_hub import create_repo, upload_folder

for repo, kind in [(MODEL_REPO,"model"), (ADAPTER_REPO,"model"), (DATASET_REPO,"dataset")]:
    try:
        create_repo(repo, repo_type=kind, exist_ok=True, private=True)
        print(f"  ✅ repo ready: {repo}")
    except Exception as e:
        print(f"  ⚠️ {repo}: {e}")

print(f"\n⬆️  uploading merged HF model…")
upload_folder(folder_path=str(MERGED_LOCAL), repo_id=MODEL_REPO, repo_type="model",
              commit_message="v2: merged 16-bit (response-only loss, EOS-terminated training)")
print(f"   ✅ https://huggingface.co/{MODEL_REPO}")

print(f"\n⬆️  uploading adapter…")
upload_folder(folder_path=str(FINAL_ADAPTER), repo_id=ADAPTER_REPO, repo_type="model",
              ignore_patterns=["merged/*", "checkpoint-*", "runs/*"],
              commit_message="v2 LoRA adapter")
print(f"   ✅ https://huggingface.co/{ADAPTER_REPO}")

print(f"\n⬆️  uploading corpus…")
upload_folder(folder_path=str(CORPUS_DIR), repo_id=DATASET_REPO, repo_type="dataset",
              ignore_patterns=["raw_*", "checkpoints/*"],
              commit_message="v1 corpus snapshot")
print(f"   ✅ https://huggingface.co/datasets/{DATASET_REPO}")

  ✅ repo ready: Shinzmann/naija-reviewer-8b-v2
  ✅ repo ready: Shinzmann/naija-reviewer-8b-v2-lora
  ✅ repo ready: Shinzmann/npa-corpus-v1

⬆️  uploading merged HF model…


No files have been modified since last commit. Skipping to prevent empty commit.


   ✅ https://huggingface.co/Shinzmann/naija-reviewer-8b-v2

⬆️  uploading adapter…
   ✅ https://huggingface.co/Shinzmann/naija-reviewer-8b-v2-lora

⬆️  uploading corpus…


No files have been modified since last commit. Skipping to prevent empty commit.


   ✅ https://huggingface.co/datasets/Shinzmann/npa-corpus-v1


## 6. GGUF — build llama.cpp

CMake build. Llama 3.1 needs a recent `convert_hf_to_gguf.py`, so we always pull master.


In [ ]:
# Disk preflight
free_gb = shutil.disk_usage(WORK_DIR).free / 1e9
print(f"Free disk at WORK_DIR: {free_gb:.1f} GB")
if free_gb < 40:
    print("⚠️  <40 GB free. f16 GGUF alone is ~16 GB. Consider freeing space.")
else:
    print("✅ enough disk")

Free disk at WORK_DIR: 167.9 GB
✅ enough disk


In [ ]:
!apt-get install -y -qq cmake build-essential >/dev/null 2>&1 || true
LLAMA_CPP = WORK_DIR / "llama.cpp"

if LLAMA_CPP.exists() and (LLAMA_CPP / "convert_hf_to_gguf.py").exists():
    print(f"✅ llama.cpp already cloned at {LLAMA_CPP}")
else:
    if LLAMA_CPP.exists():
        shutil.rmtree(LLAMA_CPP)
    print("📥 cloning llama.cpp…")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/ggerganov/llama.cpp.git", str(LLAMA_CPP)],
                   check=True)
    print("✅ cloned")

subprocess.run(["pip", "install", "-q", "-r", str(LLAMA_CPP / "requirements.txt")],
               check=True)
print("✅ converter deps installed")

📥 cloning llama.cpp…
✅ cloned
✅ converter deps installed


In [ ]:
# Build with CMake (Release). llama-quantize + llama-cli are what we need.
BUILD_DIR = LLAMA_CPP / "build"

if (BUILD_DIR / "bin" / "llama-quantize").exists() or (BUILD_DIR / "llama-quantize").exists():
    print(f"✅ llama.cpp already built at {BUILD_DIR}")
else:
    print("🔨 building llama.cpp (CMake Release) — ~3–5 min on Colab CPU…")
    t0 = time.time()
    subprocess.run(["cmake", "-B", str(BUILD_DIR), "-S", str(LLAMA_CPP),
                    "-DCMAKE_BUILD_TYPE=Release"], check=True)
    subprocess.run(["cmake", "--build", str(BUILD_DIR), "--config", "Release",
                    "-j", str(os.cpu_count() or 4)], check=True)
    print(f"✅ built in {(time.time()-t0)/60:.1f} min")

# Binary location — newer CMake versions put them in build/bin/, older in build/
def _find_bin(name: str) -> Path:
    for p in [BUILD_DIR / "bin" / name, BUILD_DIR / name]:
        if p.exists():
            return p
    raise FileNotFoundError(f"can't find {name} in {BUILD_DIR}")

LLAMA_QUANTIZE = _find_bin("llama-quantize")
LLAMA_CLI      = _find_bin("llama-cli")
CONVERT_SCRIPT = LLAMA_CPP / "convert_hf_to_gguf.py"

print(f"✅ quantize: {LLAMA_QUANTIZE}")
print(f"✅ cli:      {LLAMA_CLI}")

🔨 building llama.cpp (CMake Release) — ~3–5 min on Colab CPU…
✅ built in 3.4 min
✅ quantize: /content/gguf_work/llama.cpp/build/bin/llama-quantize
✅ cli:      /content/gguf_work/llama.cpp/build/bin/llama-cli


In [ ]:
# Fix the torchvision circular import that's breaking AutoTokenizer/AutoConfig.
# After this cell finishes: Runtime -> Restart session, then run the next cell.
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--force-reinstall", "--no-deps", "torchvision"], check=True)

print("✅ torchvision reinstalled. NOW RESTART THE RUNTIME before running the next cell.")

✅ torchvision reinstalled. NOW RESTART THE RUNTIME before running the next cell.


## 7. GGUF — convert to f16, then quantize

Three quants — Q8_0 (near-lossless), Q5_K_M (recommended quality), Q4_K_M (Ollama default).


In [ ]:
import sys
import subprocess
import time
import json
import shutil
import os
from huggingface_hub import hf_hub_download

F16_GGUF = GGUF_OUT_DIR / f"{MODEL_NAME}-f16.gguf"

if F16_GGUF.exists() and F16_GGUF.stat().st_size > 10 * 1e9:
    print(f"✅ f16 already exists ({F16_GGUF.stat().st_size/1e9:.1f} GB) — skipping")
else:
    # ---- 1. Sanity check: make sure transformers can load the model config ----
    # If this fails, the conversion will fail later anyway, so fail fast with a clear message.
    print("🔎 Sanity-checking transformers can import LlamaConfig...")
    try:
        from transformers import AutoConfig, AutoTokenizer  # noqa: F401
        _ = AutoConfig.from_pretrained(str(MERGED_LOCAL))
        print("   -> OK")
    except Exception as e:
        print(f"   -> ❌ transformers is broken: {type(e).__name__}: {e}")
        print("   -> Run the torchvision-reinstall cell, RESTART THE RUNTIME, and try again.")
        raise

    # ---- 2. Clean config.json (strip Unsloth/quantization artifacts) ----
    print("🔧 Cleaning config.json...")
    config_path = MERGED_LOCAL / "config.json"
    if config_path.exists():
        with open(config_path, "r") as f:
            cfg = json.load(f)
        removed = []
        for k in ("auto_map", "quantization_config"):
            if k in cfg:
                del cfg[k]
                removed.append(k)
        if removed:
            with open(config_path, "w") as f:
                json.dump(cfg, f, indent=2)
            print(f"   -> Removed: {', '.join(removed)}")
        else:
            print("   -> Nothing to remove")

    # ---- 3. Pull clean tokenizer files from the ungated base model ----
    print("🔧 Refreshing tokenizer files from base model...")
    base_repo = "unsloth/Meta-Llama-3.1-8B-Instruct"
    for fname in ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json"]:
        try:
            src = hf_hub_download(
                repo_id=base_repo,
                filename=fname,
                token=os.environ.get("HF_TOKEN"),
            )
            shutil.copy(src, MERGED_LOCAL / fname)
            print(f"   -> Copied {fname}")
        except Exception as e:
            print(f"   -> ⚠️ Failed to copy {fname}: {e}")

    # Remove any stray tokenizer.model — Llama 3 is BPE, not SentencePiece.
    stray = MERGED_LOCAL / "tokenizer.model"
    if stray.exists():
        stray.unlink()
        print("   -> Removed stray tokenizer.model (Llama 3 is BPE)")

    # ---- 4. Run the conversion ----
    print(f"\n🔄 converting → {F16_GGUF}")
    t0 = time.time()
    try:
        result = subprocess.run(
            [sys.executable, str(CONVERT_SCRIPT), str(MERGED_LOCAL),
             "--outfile", str(F16_GGUF), "--outtype", "f16"],
            check=True,
            capture_output=True,
            text=True,
        )
        print(f"✅ {(time.time()-t0)/60:.1f} min, {F16_GGUF.stat().st_size/1e9:.2f} GB")
    except subprocess.CalledProcessError as e:
        print("❌ Conversion failed! Error from llama.cpp:")
        print("\n=== STDERR (last 4000 chars) ===\n")
        print(e.stderr[-4000:] if e.stderr else "(empty)")
        print("\n=== STDOUT (last 2000 chars) ===\n")
        print(e.stdout[-2000:] if e.stdout else "(empty)")
        raise

🔎 Sanity-checking transformers can import LlamaConfig...
   -> OK
🔧 Cleaning config.json...
   -> Nothing to remove
🔧 Refreshing tokenizer files from base model...
   -> Copied tokenizer.json
   -> Copied tokenizer_config.json
   -> Copied special_tokens_map.json

🔄 converting → /content/gguf_work/gguf_out/naija-reviewer-8b-v2-f16.gguf
❌ Conversion failed! Error from llama.cpp:

=== STDERR (last 4000 chars) ===

external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/llama/configuration_llama.py", line 26, in <module>
    from ...utils.type_validators import interval
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/type_validators.py", line 8, in <module>
    from ..video_utils import VideoMetadataType
  File "/usr/local/lib/python3.12/dist-packages/transformers/video_utils.py", line 27, in <module>
    from .image_transforms import PaddingMo

CalledProcessError: Command '['/usr/bin/python3', '/content/gguf_work/llama.cpp/convert_hf_to_gguf.py', '/content/gguf_work/merged_hf', '--outfile', '/content/gguf_work/gguf_out/naija-reviewer-8b-v2-f16.gguf', '--outtype', 'f16']' returned non-zero exit status 1.

In [ ]:
import sys
import subprocess
import time
import json
import shutil
import os
from huggingface_hub import hf_hub_download

F16_GGUF = GGUF_OUT_DIR / f"{MODEL_NAME}-f16.gguf"

if F16_GGUF.exists() and F16_GGUF.stat().st_size > 10 * 1e9:
    print(f"✅ f16 already exists ({F16_GGUF.stat().st_size/1e9:.1f} GB) — skipping")
else:
    print("🔧 Fixing config.json and tokenizer for llama.cpp compatibility...")

    # 1. Remove auto_map from config.json if Unsloth added it
    config_path = MERGED_LOCAL / "config.json"
    if config_path.exists():
        with open(config_path, "r") as f:
            cfg = json.load(f)
        if "auto_map" in cfg:
            print("   -> Removed auto_map from config.json")
            del cfg["auto_map"]
            with open(config_path, "w") as f:
                json.dump(cfg, f, indent=2)

    # 2. Pull clean tokenizer configs from an ungated base model
    base_repo = "unsloth/Meta-Llama-3.1-8B-Instruct"
    for fname in ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json"]:
        try:
            src = hf_hub_download(repo_id=base_repo, filename=fname, token=os.environ.get("HF_TOKEN"))
            shutil.copy(src, MERGED_LOCAL / fname)
            print(f"   -> Copied {fname} from {base_repo}")
        except Exception as e:
            print(f"   -> Failed to copy {fname}: {e}")

    print(f"\n🔄 converting → {F16_GGUF}")
    t0 = time.time()
    try:
        subprocess.run(
            [sys.executable, str(CONVERT_SCRIPT), str(MERGED_LOCAL),
             "--outfile", str(F16_GGUF), "--outtype", "f16"],
            check=True,
            capture_output=True,
            text=True
        )
        print(f"✅ {(time.time()-t0)/60:.1f} min, {F16_GGUF.stat().st_size/1e9:.2f} GB")
    except subprocess.CalledProcessError as e:
        print("❌ Conversion failed! Here is the error from llama.cpp:")
        print("\n=== STDERR ===\n")
        print(e.stderr)
        print("\n=== STDOUT ===\n")
        print(e.stdout)
        raise

In [ ]:
QUANT_LEVELS = ["Q4_K_M", "Q5_K_M", "Q8_0"]   # add Q3_K_M / Q6_K / Q2_K if you want

def quantize(level: str) -> Path:
    out = GGUF_OUT_DIR / f"{MODEL_NAME}-{level}.gguf"
    if out.exists() and out.stat().st_size > 1e9:
        print(f"  ✅ {level} exists ({out.stat().st_size/1e9:.2f} GB) — skipping")
        return out
    print(f"  🔧 quantizing → {level}")
    t0 = time.time()
    subprocess.run([str(LLAMA_QUANTIZE), str(F16_GGUF), str(out), level], check=True)
    print(f"     ✅ {(time.time()-t0)/60:.1f} min, {out.stat().st_size/1e9:.2f} GB")
    return out

print(f"🔄 quantizing to {len(QUANT_LEVELS)} levels…")
quant_paths = {lvl: quantize(lvl) for lvl in QUANT_LEVELS}
print("\n✅ all quants done")
for lvl, p in quant_paths.items():
    print(f"   {lvl:8s}  {p.stat().st_size/1e9:5.2f} GB  {p.name}")

## 8. Smoke-test the GGUF

Confirm Q4_K_M loads and generates sensibly (and stops cleanly thanks to the EOS fix).


In [ ]:
TEST_QUANT = quant_paths.get("Q4_K_M") or next(iter(quant_paths.values()))

PROMPT = '''### Instruction
Write a short review of the product described below.

### Input
A pair of wireless earbuds with 30-hour battery life, noise cancellation, and IPX5 water resistance.

### Response
'''

print(f"🧪 testing {TEST_QUANT.name}\n")
result = subprocess.run(
    [str(LLAMA_CLI), "-m", str(TEST_QUANT), "-p", PROMPT,
     "-n", "200", "--temp", "0.7", "--top-p", "0.9",
     "-no-cnv", "--no-display-prompt"],
    capture_output=True, text=True, timeout=300,
)
print("─── output ───")
print(result.stdout.strip())
if result.returncode != 0:
    print("\n─── stderr (tail) ───")
    print(result.stderr[-1500:])

## 9. Generate `Modelfile` + `README.md`

Modelfile encodes the **Alpaca template** (not Llama-3.1 chat) so Ollama prompts the model the way you trained it. README provides usage examples on the HF page.


In [ ]:
# Modelfile — placeholder substitution avoids f-string collision with Go template syntax
_MODELFILE_TMPL = """FROM ./__MODEL_FILENAME__

# Alpaca prompt template — matches the format used during fine-tuning.
TEMPLATE \"\"\"### Instruction
{{ if .System }}{{ .System }}{{ else }}Provide a helpful response.{{ end }}

### Input
{{ .Prompt }}

### Response
\"\"\"

PARAMETER stop "### Instruction"
PARAMETER stop "### Input"
PARAMETER stop "### Response"
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 4096

SYSTEM "You are NaijaReviewer, an AI trained on Nigerian product reviews. Write thoughtful, culturally-aware reviews."
"""

MODELFILE_CONTENT = _MODELFILE_TMPL.replace("__MODEL_FILENAME__", f"{MODEL_NAME}-Q4_K_M.gguf")
MODELFILE_PATH = GGUF_OUT_DIR / "Modelfile"
MODELFILE_PATH.write_text(MODELFILE_CONTENT)
print(f"✅ wrote {MODELFILE_PATH}")
print("\n─── Modelfile ───")
print(MODELFILE_CONTENT)

In [ ]:
# Model card — rendered on the HF repo page
def _quant_table_md():
    sizes = {lvl: p.stat().st_size / 1e9 for lvl, p in quant_paths.items()}
    notes = {
        "Q2_K":   "smallest, significant quality loss",
        "Q3_K_M": "very small, noticeable quality loss",
        "Q4_K_M": "**recommended** — balanced size/quality (Ollama default)",
        "Q5_K_M": "**recommended** for quality — slightly larger",
        "Q6_K":   "very low quality loss",
        "Q8_0":   "near-lossless, ~2× size of Q4_K_M",
    }
    rows = ["| File | Quant | Size | Notes |", "|---|---|---|---|"]
    for lvl in sorted(sizes.keys()):
        fname = f"{MODEL_NAME}-{lvl}.gguf"
        rows.append(f"| `{fname}` | `{lvl}` | {sizes[lvl]:.2f} GB | {notes.get(lvl, '')} |")
    return "\n".join(rows)

README_CONTENT = f"""---
license: llama3.1
base_model: meta-llama/Meta-Llama-3.1-8B-Instruct
library_name: gguf
pipeline_tag: text-generation
tags:
- gguf
- llama-3.1
- llama-cpp
- ollama
- quantized
- lora
- naija
- nigerian
- reviews
---

# {MODEL_NAME}

Llama-3.1-8B-Instruct fine-tuned with QLoRA on Nigerian product-review data.
Both **HF format** (for `transformers` / Unsloth) and **GGUF** (for llama.cpp / Ollama)
files are in this repo.

## What's in the repo

- HF model files (`*.safetensors`, `config.json`, tokenizer) — load with `transformers`
- GGUF builds (see table below) — load with `llama.cpp` / Ollama
- `Modelfile` — ready-to-use Ollama configuration with the correct Alpaca template

## Available GGUF quantizations

{_quant_table_md()}

## Quick start — Ollama

```bash
huggingface-cli download {MODEL_REPO} {MODEL_NAME}-Q4_K_M.gguf Modelfile --local-dir .
ollama create {MODEL_NAME} -f Modelfile
ollama run {MODEL_NAME} "Write a review of jollof rice from a buka in Surulere."
```

## Quick start — llama.cpp

```bash
huggingface-cli download {MODEL_REPO} {MODEL_NAME}-Q4_K_M.gguf --local-dir .
./llama-cli -m {MODEL_NAME}-Q4_K_M.gguf \\
    -p "### Instruction\\nWrite a product review.\\n\\n### Input\\nA Tecno Spark 10 Pro phone.\\n\\n### Response\\n" \\
    -n 256 --temp 0.7 -no-cnv
```

## Quick start — `llama-cpp-python`

```python
from llama_cpp import Llama
llm = Llama.from_pretrained(
    repo_id="{MODEL_REPO}",
    filename="{MODEL_NAME}-Q4_K_M.gguf",
    n_ctx=4096,
)
out = llm(
    "### Instruction\\nWrite a review.\\n\\n### Input\\nA pair of wireless earbuds.\\n\\n### Response\\n",
    max_tokens=256, temperature=0.7, stop=["### Instruction"],
)
print(out["choices"][0]["text"])
```

## Quick start — `transformers` (HF format)

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tok = AutoTokenizer.from_pretrained("{MODEL_REPO}")
model = AutoModelForCausalLM.from_pretrained("{MODEL_REPO}", torch_dtype=torch.bfloat16, device_map="auto")
prompt = "### Instruction\\nWrite a review.\\n\\n### Input\\nA Tecno phone.\\n\\n### Response\\n"
out = model.generate(**tok(prompt, return_tensors="pt").to(model.device), max_new_tokens=200, eos_token_id=tok.eos_token_id)
print(tok.decode(out[0], skip_special_tokens=True))
```

## Prompt format

Fine-tuned on the **Alpaca template**:

```
### Instruction
{{instruction}}

### Input
{{input}}

### Response
{{response}}
```

The included `Modelfile` already encodes this for Ollama.

## Training details

- **Base:** `meta-llama/Meta-Llama-3.1-8B-Instruct`
- **Method:** QLoRA (4-bit) via [Unsloth](https://github.com/unslothai/unsloth)
- **LoRA:** r=16, alpha=32, dropout=0.1, all 7 linear targets
- **Epochs:** 2
- **v2 improvements over v1:**
  - EOS token appended to training examples (model learns to stop)
  - Response-only loss masking via `train_on_responses_only` (gradients on the answer, not the prompt scaffolding)

## License

Subject to the [Llama 3.1 Community License](https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct/blob/main/LICENSE).
"""

README_PATH = GGUF_OUT_DIR / "README.md"
README_PATH.write_text(README_CONTENT)
print(f"✅ wrote {README_PATH}  ({len(README_CONTENT):,} chars)")

## 10. Upload GGUFs + Modelfile + README to the same repo

Files go up one at a time — a dropped connection mid-upload doesn't restart 30 GB.


In [ ]:
from huggingface_hub import HfApi
api = HfApi()

def _upload(local_path: Path, repo_path: str):
    size_gb = local_path.stat().st_size / 1e9
    print(f"  ⬆️  {local_path.name:42s} ({size_gb:5.2f} GB) → {repo_path}")
    t0 = time.time()
    api.upload_file(path_or_fileobj=str(local_path), path_in_repo=repo_path,
                    repo_id=MODEL_REPO, repo_type="model",
                    commit_message=f"Add {repo_path}")
    print(f"     ✅ {(time.time()-t0)/60:.1f} min")

# GGUFs
for lvl, p in quant_paths.items():
    _upload(p, p.name)

# Modelfile + README (README OVERWRITES any one auto-generated by HF)
_upload(MODELFILE_PATH, "Modelfile")
_upload(README_PATH,    "README.md")

print(f"\n🎉 done → https://huggingface.co/{MODEL_REPO}")
print("   (currently private — flip to public in settings when ready)")

## 11. Optional — free up local disk


In [ ]:
# Uncomment to clear local workspace (Drive copies are untouched)
# shutil.rmtree(WORK_DIR, ignore_errors=True)
# print(f"🗑️  removed {WORK_DIR}")
print("(no-op — uncomment to clean up)")

## 12. Local usage cheatsheet

On your laptop:

```bash
huggingface-cli download Shinzmann/naija-reviewer-8b-v2 naija-reviewer-8b-v2-Q4_K_M.gguf Modelfile --local-dir .
ollama create naija-reviewer-8b-v2 -f Modelfile
ollama run naija-reviewer-8b-v2
```

To wire into your FastAPI demo, update `.env`:
```
TASK1_BACKBONE=ollama:naija-reviewer-8b-v2
```

Then restart `make demo`. FastAPI now serves the v2 fine-tune at zero API cost.

---

🎉 **All done.** End-to-end: corpus → fine-tune (response-only loss, EOS-terminated) → merge → smoke test → push HF → GGUF (f16 + 3 quants) → push GGUFs + Modelfile + README → ready to run locally.
